# 10 — Final Ensemble (CTST × ECOD-plus-temporal)
Rank-average the joint-structure model (CTST, notebook 09) with the best per-feature tails model (`submission_ecod_plus_temporal.csv`, 95.28%).

**Why this should work better than either alone:** ECOD scores by independent per-feature tails — it catches single-feature extremes but misses anomalies that are normal-looking marginally yet anomalous jointly. CTST scores joint distribution shift directly. The two view different facets of the anomaly; ranks should be partially uncorrelated, so rank-averaging recovers both.

**Why rank-average and not weighted score-average:** the two scores live on different scales (ECOD sigmoid-calibrated vs XGBoost probability). Ranks are scale-free and AUC-invariant. Equal weight: no prior reason to prefer one model.

Writes `submission_final.csv`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.stats import rankdata

out = ROOT / 'outputs' / 'submissions'
ctst = pd.read_csv(out / 'submission_ctst.csv').sort_values('ID').reset_index(drop=True)
ecod = pd.read_csv(out / 'submission_ecod_plus_temporal.csv').sort_values('ID').reset_index(drop=True)
assert (ctst['ID'].to_numpy() == ecod['ID'].to_numpy()).all()
print('rows:', len(ctst))

rows: 289101


In [2]:
from numpy import corrcoef
r_ctst = rankdata(ctst['TARGET'].to_numpy())
r_ecod = rankdata(ecod['TARGET'].to_numpy())
print('rank correlation:', round(float(corrcoef(r_ctst, r_ecod)[0, 1]), 4))
n = len(r_ctst)
score = (r_ctst + r_ecod) / (2 * n)
print('min/max/mean:', round(float(score.min()), 4), round(float(score.max()), 4), round(float(score.mean()), 4))

rank correlation: -0.3082
min/max/mean: 0.0571 0.9913 0.5


In [3]:
pd.DataFrame({'ID': ctst['ID'], 'TARGET': score}).to_csv(out / 'submission_final.csv', index=False)